# 25 · Local research API contract and smoke test

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Development API only. No public tunnel, authentication bypass, remote participant upload or medical release is created.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Install the local API dependencies

In [ ]:
import subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)+'[app]'],check=True)

## 2. Load a locally exported research model

In [ ]:
from oncoplate.api import create_app
EXPORT_DIR=p['exports']/'resnet50_frozen_joint_s0'
assert (EXPORT_DIR/'export_contract.json').exists(),'Complete notebook 15 first.'
app=create_app(EXPORT_DIR)
print(app.title)

## 3. Exercise the API in-process with a local validation image

In [ ]:
from fastapi.testclient import TestClient
from oncoplate.pipeline import load_study
records,_=load_study(cfg,'joint',stage_images=True)
image=Path(records[records.split.eq('validation')].image_path.iloc[0])
with TestClient(app) as client:
    print(client.get('/health').json())
    response=client.post('/research/predict',files={'image':(image.name,image.read_bytes(),'image/jpeg')})
    print(response.status_code,response.json())

## 4. Save the platform-neutral contract

In [ ]:
write_json(p['exports']/'research_api_openapi.json',app.openapi())
print('The API returns qualified visual predictions only. It does not claim to identify chemical concentration, cancer probability or safe foods.')
print('For the full study, use the reviewed source/PCSI engine; mobile and user-study release gates remain separate.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
